In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%%RecordEvent
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd

from pathlib import Path

In [ ]:
%%RecordEvent
%%time
### cell 0 ###

benchmark_name = "what-course-are-you-going-to-take"
course = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "course-reviews-university-of-waterloo" / "course_data_clean.csv"
)
factor = 30
course = pd.concat([course] * factor, ignore_index=True)
course.info()

In [ ]:
%%RecordEvent
%%time
### cell 1 ###

course.head(10)

In [ ]:
%%RecordEvent
%%time
### cell 2 ###

course.tail(10)

In [ ]:
%%RecordEvent
%%time
### cell 3 ###

course.describe()

In [ ]:
%%RecordEvent
%%time
### cell 4 ###

course.info()

In [ ]:
%%RecordEvent
%%time
### cell 5 ###

course = course.dropna()

In [ ]:
%%RecordEvent
%%time
### cell 6 ###

course[["course_unit", "course_num"]] = course["course_code"].str.split(
    " ", expand=True
)

In [ ]:
%%RecordEvent
%%time
### cell 7 ###

course

In [ ]:
%%RecordEvent
%%time
### cell 8 ###

course[course["num_reviews"] < 10].index

In [ ]:
%%RecordEvent
%%time
### cell 9 ###


def extract_filter_features(df, col_name, threshold):
    """Extracts features for predicting execution time of filtering operation."""

    num_rows = len(df)
    dtype_str = str(df[col_name].dtype)  # Get dtype as string
    nan_ratio = df[col_name].isna().mean()  # Compute NaN ratio

    # Compute selectivity: fraction of rows passing the filter
    if df[col_name].dropna().empty:
        target_selectivity = 0.0
    else:
        target_selectivity = (df[col_name] < threshold).mean()

    return {
        "num_rows": num_rows,
        "dtype_str": dtype_str,
        "nan_ratio": nan_ratio,
        "target_selectivity": target_selectivity,
    }


extract_filter_features(course, "num_reviews", 10)

In [ ]:
%%RecordEvent
%%time
### cell 10 ###

course.drop(course[course["num_reviews"] < 10].index, inplace=True)

In [ ]:
%%RecordEvent
%%time
### cell 11 ###

course

In [ ]:
%%RecordEvent
%%time
### cell 12 ###

for i_1 in ["useful", "easy", "liked"]:
    course[i_1] = course[i_1].str.replace("%", "")
    course[i_1] = course[i_1].astype("int")

In [ ]:
%%RecordEvent
%%time
### cell 13 ###

course.set_index("course_unit", inplace=True)

In [ ]:
%%RecordEvent
%%time
### cell 14 ###

course

In [ ]:
%%RecordEvent
%%time
### cell 15 ###

course.drop(["course_title", "reviews", "course_rating"], axis=1, inplace=True)

In [ ]:
%%RecordEvent
%%time
### cell 16 ###

course

In [ ]:
%%RecordEvent
%%time
### cell 17 ###

course.info()

In [ ]:
%%RecordEvent
%%time
### cell 18 ###

course_gp = course.groupby("course_unit").mean(numeric_only=True)

In [ ]:
%%RecordEvent
%%time
### cell 19 ###

course_gp.head()

In [ ]:
%%RecordEvent
%%time
### cell 20 ###

course["course_rating_mean"] = None
for i_2 in course_gp.index:
    course.loc[i_2, "course_rating_mean"] = course_gp.loc[i_2, "course_rating_int"]

In [ ]:
%%RecordEvent
%%time
### cell 21 ###

course

In [ ]:
%%RecordEvent
%%time
### cell 22 ###

course.reset_index(inplace=True)

In [ ]:
%%RecordEvent
%%time
### cell 23 ###

course

In [ ]:
%%RecordEvent
%%time
### cell 24 ###

course.groupby("course_unit").mean(numeric_only=True)["course_rating_int"]

In [ ]:
%%RecordEvent
%%time
### cell 25 ###


def extract_features_from_dataframe(df, group_column, agg_function="mean"):
    """Extracts features for predicting the execution time of a groupby operation."""
    n_rows = df.shape[0]
    n_cols = df.shape[1]

    n_groups = df[group_column].nunique()
    group_sizes = df[group_column].value_counts()
    max_group_size = group_sizes.max()

    int_count = df.select_dtypes(include=["int", "int32", "int64"]).shape[1]
    float_count = df.select_dtypes(include=["float", "float32", "float64"]).shape[1]
    str_count = df.select_dtypes(include=["object", "string"]).shape[1]

    aggregation_mapping = {"sum": 0, "mean": 1, "count": 2}
    agg_function_encoded = aggregation_mapping.get(agg_function, 1)  # Default to 'mean'

    return {
        "n_rows": n_rows,
        "n_cols": n_cols,
        "group_range": n_groups,  # Alias for n_groups
        "n_groups": n_groups,
        "max_group_size": max_group_size,
        "int_count": int_count,
        "float_count": float_count,
        "str_count": str_count,
        "agg_function": agg_function_encoded,
    }


extract_features_from_dataframe(course, "course_unit", "mean")

In [ ]:
%%RecordEvent
%%time
### cell 26 ###

course[course["course_code"].str.startswith("CS")].value_counts()

In [ ]:
%%RecordEvent
%%time
### cell 27 ###

course

In [ ]:
%%RecordEvent
%%time
### cell 28 ###

course.sort_values("course_rating_mean", ascending=False, inplace=True)

In [ ]:
%%RecordEvent
%%time
### cell 29 ###

course.reset_index(inplace=True)

In [ ]:
%%RecordEvent
%%time
### cell 30 ###

course.set_index("course_unit", inplace=True)

In [ ]:
%%RecordEvent
%%time
### cell 31 ###

course.loc["KOREA", "course_rating_mean"].value_counts()

In [ ]:
%%RecordEvent
%%time
### cell 32 ###

KOREA = course.loc["KOREA", :]

In [ ]:
%%RecordEvent
%%time
### cell 33 ###

course.loc["CHINA", "course_rating_mean"].value_counts()

In [ ]:
%%RecordEvent
%%time
### cell 34 ###

china = course.loc["CHINA", :]

In [ ]:
%%RecordEvent
%%time
### cell 35 ###

course.loc["CHINA", "course_rating_mean"].value_counts()

In [ ]:
%%RecordEvent
%%time
### cell 36 ###

span = course.loc["SPAN", :]

In [ ]:
%%RecordEvent
%%time
### cell 37 ###

course.loc["CS", "course_rating_mean"].value_counts()

In [ ]:
%%RecordEvent
%%time
### cell 38 ###

cs = course.loc["CS", :]
cs

In [ ]:
%%RecordEvent
%%time
### cell 39 ###

cs_mean = (
    cs.groupby("course_code")
    .mean(numeric_only=True)
    .sort_values("course_rating_int", ascending=False)
)
cs_mean

In [ ]:
%%RecordEvent
%%time
### cell 40 ###

course.loc["WKRPT", "course_rating_mean"].value_counts()

In [ ]:
%%RecordEvent
%%time
### cell 41 ###

wkrpt = course.loc["WKRPT", :]
wkrpt

In [ ]:
%%RecordEvent
%%time
### cell 42 ###

wkrpt_mean = (
    wkrpt.groupby("course_code")
    .mean(numeric_only=True)
    .sort_values("course_rating_int", ascending=False)
)
wkrpt_mean

In [ ]:
%%RecordEvent
%%time
### cell 43 ###

wkrpt.groupby("course_code").mean(numeric_only=True)

In [ ]:
%%RecordEvent
%%time
### cell 44 ###

course.loc["PD", "course_rating_mean"].value_counts()

In [ ]:
%%RecordEvent
%%time
### cell 45 ###

pd_df = course.loc["PD", :]
pd_df

In [ ]:
%%RecordEvent
%%time
### cell 46 ###

pd_mean = (
    pd_df.groupby("course_code")
    .mean(numeric_only=True)
    .sort_values("course_rating_int", ascending=False)
)
pd_mean